# 1. Project Setup and Dataset Loading

This notebook implements the expanded classical NLP baseline for NutriVision using the Food.com recipe dataset.

The objective is to develop an ingredient-aware recipe retrieval system using classical NLP techniques and evaluate different text representations.

The prepared Food.com master dataset serves as the reference corpus for the retrieval experiments.

In [1]:
# --------------------------------------------------
# 1. Project Setup and Dataset Loading
# --------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
PROJECT_ROOT = Path(r"C:\Users\dell\NutriVision")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "recipes"
    / "foodcom_recipes_master.csv"
)

# Load prepared Food.com dataset
recipes_df = pd.read_csv(DATA_PATH)

print("Dataset shape:", recipes_df.shape)
print("\nAvailable columns:")
print(recipes_df.columns.tolist())

print("\nFirst 3 recipes:")
display(
    recipes_df[
        [
            "recipe_id",
            "recipe_name",
            "recipe_text_core",
            "recipe_text_with_description"
        ]
    ].head(3)
)

Dataset shape: (231637, 23)

Available columns:
['recipe_name', 'recipe_id', 'preparation_time_min', 'contributor_id', 'submitted', 'tags', 'nutrition', 'step_count', 'steps', 'description', 'ingredients', 'ingredient_count', 'calories_kcal', 'has_description', 'has_steps', 'has_ingredients', 'recipe_text', 'recipe_text_core', 'recipe_text_with_description', 'has_minimum_ingredients', 'has_preparation_steps', 'has_multiple_steps', 'has_valid_core_text']

First 3 recipes:


,recipe_id,recipe_name,recipe_text_core,recipe_text_with_description
0,137739,arriba baked winter squash mexican style,Recipe: arriba baked winter squash mexican s...,Recipe: arriba baked winter squash mexican s...
1,31490,a bit different breakfast pizza,Recipe: a bit different breakfast pizza. Ingr...,Recipe: a bit different breakfast pizza. Desc...
2,112140,all in the kitchen chili,Recipe: all in the kitchen chili. Ingredients...,Recipe: all in the kitchen chili. Description...


### 2.1 Text Preprocessing

The objective of this stage is to prepare recipe text for classical NLP representations and ingredient-aware recipe retrieval.

Preprocessing decisions will be based on the characteristics of the Food.com dataset. In particular, we want to preserve meaningful food-related terms, ingredient names, quantities, and preparation information while removing unnecessary textual noise.

We will first inspect representative recipes before defining and applying the preprocessing pipeline.

In [2]:
# --------------------------------------------------
# 2.1.1 Inspect Representative Recipes
# --------------------------------------------------

# Select recipes with different characteristics
sample_recipes = recipes_df.sample(
    n=5,
    random_state=42
)

for _, recipe in sample_recipes.iterrows():

    print("=" * 80)
    print("Recipe ID:", recipe["recipe_id"])
    print("Recipe name:", recipe["recipe_name"])

    print("\nIngredients:")
    print(recipe["ingredients"])

    print("\nPreparation steps:")
    print(recipe["steps"])

    print("\nDescription:")
    print(recipe["description"])

    print("\nCore text:")
    print(recipe["recipe_text_core"])

    print("\n")

Recipe ID: 94947
Recipe name: crab filled crescent snacks

Ingredients:
['crabmeat', 'cream cheese', 'green onions', 'garlic salt', 'refrigerated crescent dinner rolls', 'egg yolk', 'water', 'sesame seeds', 'sweet and sour sauce']

Preparation steps:
['heat over to 375 degrees', 'spray large cookie sheet with non-stick cooking spray', 'in small bowl , combine crabmeat , cream cheese , onions and garlic salt and mix well', 'unroll both cans of dough', 'separate into 16 triangles', 'cut each triangle in half lengthwise to make 32 triangles', 'place 1 teaspoon crab mixture on center of each triangle about 1 inch from short side of triangle', 'fold short ends of each triangle over filling', 'pinch sides to seal', 'roll up', 'place on sprayed cookie sheet', 'in small bowl , combine egg yolk and water and mix well', 'brush egg mixture over snacks', 'sprinkle with sesame seed', 'bake at 375 degrees for 15 to 20 minutes or until golden brown', 'serve warn snacks with sweet-and-sour sauce']

De

In [3]:
# --------------------------------------------------
# 2.1.1 Verify Structured Data Types
# --------------------------------------------------

sample = recipes_df.iloc[0]

print("Recipe name type:", type(sample["recipe_name"]))
print("Ingredients type:", type(sample["ingredients"]))
print("Steps type:", type(sample["steps"]))

print("\nIngredients example:")
print(sample["ingredients"])

print("\nSteps example:")
print(sample["steps"])

print("\nExisting core text:")
print(sample["recipe_text_core"][:500])

Recipe name type: <class 'str'>
Ingredients type: <class 'str'>
Steps type: <class 'str'>

Ingredients example:
['winter squash', 'mexican seasoning', 'mixed spice', 'honey', 'butter', 'olive oil', 'salt']

Steps example:
['make a choice and proceed with recipe', 'depending on size of squash , cut into half or fourths', 'remove seeds', 'for spicy squash , drizzle olive oil or melted butter over each cut squash piece', 'season with mexican seasoning mix ii', 'for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece', 'season with sweet mexican spice mix', 'bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin', 'be careful not to burn the squash especially if you opt to use sugar or butter', 'if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking', 'if desired , season with salt']

Existing core text:
Recipe: arriba   baked wi

### 2.1.2 Restore Structured Recipe Fields

Since CSV does not preserve Python list objects, the `ingredients` and `steps` columns are loaded as strings.

We will convert these columns back into Python lists using safe literal parsing. Then, we will reconstruct the recipe text using the actual ingredient and preparation-step elements.

This ensures that each ingredient and preparation instruction remains intact during subsequent NLP preprocessing.

In [4]:
# --------------------------------------------------
# 2.1.2 Restore Lists and Rebuild Recipe Text
# --------------------------------------------------

import ast

# Convert string representations of lists into Python lists
for column in ["ingredients", "steps"]:
    recipes_df[column] = recipes_df[column].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# Rebuild the retrieval text correctly
def build_recipe_text(row):
    ingredients_text = " ".join(row["ingredients"])
    steps_text = " ".join(row["steps"])

    return (
        f"Recipe: {row['recipe_name']}. "
        f"Ingredients: {ingredients_text}. "
        f"Preparation: {steps_text}"
    )

recipes_df["recipe_text_core"] = recipes_df.apply(
    build_recipe_text,
    axis=1
)

# Verify
print("Ingredients type:", type(recipes_df.iloc[0]["ingredients"]))
print("Steps type:", type(recipes_df.iloc[0]["steps"]))

print("\nCorrected recipe text:")
print(recipes_df.iloc[0]["recipe_text_core"])

Ingredients type: <class 'list'>
Steps type: <class 'list'>

Corrected recipe text:
Recipe: arriba   baked winter squash mexican style. Ingredients: winter squash mexican seasoning mixed spice honey butter olive oil salt. Preparation: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece season with sweet mexican spice mix bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin be careful not to burn the squash especially if you opt to use sugar or butter if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking if desired , season with salt


### 2.1.3 Text Quality Inspection

Before applying normalization, we will examine the reconstructed recipe corpus for:

- Missing or empty text.
- Unusual characters and punctuation.
- Numeric information, including quantities, temperatures, and cooking times.
- Differences in text length.

These characteristics will help us determine which information should be preserved during preprocessing.

In [5]:
# --------------------------------------------------
# 2.1.3 Text Quality Inspection
# --------------------------------------------------

import re

# Basic text statistics
recipes_df["text_word_count"] = (
    recipes_df["recipe_text_core"].str.split().str.len()
)

print("Missing texts:", recipes_df["recipe_text_core"].isna().sum())
print("Empty texts:", recipes_df["recipe_text_core"].str.strip().eq("").sum())

print("\nWord count statistics:")
display(recipes_df["text_word_count"].describe())

# Inspect punctuation and special characters
special_chars = recipes_df["recipe_text_core"].str.findall(
    r"[^a-zA-Z0-9\s]"
)

special_char_counts = (
    pd.Series(
        [char for chars in special_chars for char in chars]
    )
    .value_counts()
)

print("\nMost frequent special characters:")
display(special_char_counts.head(20))

# Inspect numeric information
numeric_examples = recipes_df[
    recipes_df["recipe_text_core"].str.contains(
        r"\d", regex=True, na=False
    )
]

print("\nRecipes containing numeric information:")
display(
    numeric_examples[
        ["recipe_id", "recipe_name", "recipe_text_core"]
    ].head(5)
)

Missing texts: 0
Empty texts: 0

Word count statistics:


count    231637.000000
mean        125.944927
std          77.569833
min           8.000000
25%          75.000000
50%         109.000000
75%         157.000000
max        2279.000000
Name: text_word_count, dtype: float64


Most frequent special characters:


,    1284710
:     736688
.     463913
-     310410
/     115122
'      39979
&      34754
"      28847
!      28333
)       2665
%       2437
(       1232
#        996
~        988
@        700
[        589
_        588
]        575
=        533
+        517
Name: count, dtype: int64


Recipes containing numeric information:


,recipe_id,recipe_name,recipe_text_core
0,137739,arriba baked winter squash mexican style,Recipe: arriba baked winter squash mexican s...
1,31490,a bit different breakfast pizza,Recipe: a bit different breakfast pizza. Ingr...
2,112140,all in the kitchen chili,Recipe: all in the kitchen chili. Ingredients...
3,59389,alouette potatoes,Recipe: alouette potatoes. Ingredients: sprea...
4,44061,amish tomato ketchup for canning,Recipe: amish tomato ketchup for canning. In...


### 2.1.4 Numeric Information Analysis

Numbers in recipes can represent quantities, cooking temperatures, durations, dimensions, or other preparation details.

We will inspect common numeric patterns before deciding whether numbers should be retained, normalized, or removed during preprocessing.

In [6]:
# --------------------------------------------------
# 2.1.4 Numeric Information Analysis
# --------------------------------------------------

# Extract numeric expressions from recipe texts
numeric_patterns = recipes_df["recipe_text_core"].str.findall(
    r"\d+(?:\.\d+)?(?:\s*/\s*\d+)?"
)

# Count recipes containing numbers
recipes_with_numbers = numeric_patterns.apply(len).gt(0).sum()

print("Recipes containing numbers:", recipes_with_numbers)
print(
    "Percentage:",
    round(recipes_with_numbers / len(recipes_df) * 100, 2),
    "%"
)

# Display 10 examples with their extracted numbers
numeric_examples = recipes_df.loc[
    numeric_patterns.apply(len).gt(0),
    ["recipe_id", "recipe_name", "recipe_text_core"]
].head(10).copy()

numeric_examples["extracted_numbers"] = numeric_patterns[
    numeric_examples.index
]

display(numeric_examples)

Recipes containing numbers: 198770
Percentage: 85.81 %


,recipe_id,recipe_name,recipe_text_core,extracted_numbers
0,137739,arriba baked winter squash mexican style,Recipe: arriba baked winter squash mexican s...,"[350, 40]"
1,31490,a bit different breakfast pizza,Recipe: a bit different breakfast pizza. Ingr...,"[425, 12, 5, 15, 20]"
2,112140,all in the kitchen chili,Recipe: all in the kitchen chili. Ingredients...,"[2, 3, 6, 8]"
3,59389,alouette potatoes,Recipe: alouette potatoes. Ingredients: sprea...,"[8, 8, 2, 2, 350, 15]"
4,44061,amish tomato ketchup for canning,Recipe: amish tomato ketchup for canning. In...,"[2, 1 / 2]"
5,5289,apple a day milk shake,Recipe: apple a day milk shake. Ingredients: ...,[2]
6,25274,aww marinated olives,Recipe: aww marinated olives. Ingredients: fe...,"[1, 2]"
7,67888,backyard style barbecued ribs,Recipe: backyard style barbecued ribs. Ingred...,"[1, 1, 1, 1, 350, 1, 30, 3, 1, 2]"
8,70971,bananas 4 ice cream pie,Recipe: bananas 4 ice cream pie. Ingredients:...,"[4, 9, 1, 1 / 4]"
9,75452,beat this banana bread,Recipe: beat this banana bread. Ingredients: ...,"[350, 9, 5, 45, 55, 30]"


### 2.1.5 Text Normalization and Cleaning

The initial normalization stage applies the following transformations:

1. Convert text to lowercase.
2. Normalize hyphens into spaces.
3. Remove punctuation and special characters while preserving numbers.
4. Normalize repeated whitespace.

Numbers are retained because they can represent meaningful recipe information, including ingredient quantities, cooking temperatures, and preparation times.

In [7]:
# --------------------------------------------------
# 2.1.5 Basic Text Normalization
# --------------------------------------------------

import re
import string

def normalize_text(text):

    # Convert to lowercase
    text = text.lower()

    # Replace hyphens with spaces
    text = text.replace("-", " ")

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply normalization
recipes_df["normalized_text"] = (
    recipes_df["recipe_text_core"].apply(normalize_text)
)


# Compare original and normalized examples
for idx in [0, 1, 2]:

    print("=" * 80)
    print("Recipe:", recipes_df.loc[idx, "recipe_name"])

    print("\nOriginal text:")
    print(recipes_df.loc[idx, "recipe_text_core"][:700])

    print("\nNormalized text:")
    print(recipes_df.loc[idx, "normalized_text"][:700])

Recipe: arriba   baked winter squash mexican style

Original text:
Recipe: arriba   baked winter squash mexican style. Ingredients: winter squash mexican seasoning mixed spice honey butter olive oil salt. Preparation: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece season with sweet mexican spice mix bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin be careful not to burn the squash especially if you opt to use sugar or bu

Normalized text:
recipe arriba baked winter squash mexican style ingredients winter squash mexican seasoning mixed spice honey butter olive oil salt preparation make a choice and proceed with recipe depending on size of squash cut 

### 2.1.6 Tokenization

Tokenization converts normalized recipe text into individual word tokens.

We will initially use whitespace-based tokenization because punctuation has already been normalized. This provides a simple and reproducible starting point for subsequent classical NLP representations.

At this stage, no stopwords or food-related terms are removed.

In [8]:
# --------------------------------------------------
# 2.1.6 Tokenization
# --------------------------------------------------

# Tokenize normalized recipe text
recipes_df["tokens"] = recipes_df["normalized_text"].str.split()

# Inspect representative examples
for idx in [0, 1, 2]:

    print("=" * 80)
    print("Recipe:", recipes_df.loc[idx, "recipe_name"])

    print("\nTokens:")
    print(recipes_df.loc[idx, "tokens"])

    print("\nToken count:")
    print(len(recipes_df.loc[idx, "tokens"]))


# Overall token statistics
recipes_df["token_count"] = recipes_df["tokens"].str.len()

print("\nOverall token statistics:")
display(recipes_df["token_count"].describe())

Recipe: arriba   baked winter squash mexican style

Tokens:
['recipe', 'arriba', 'baked', 'winter', 'squash', 'mexican', 'style', 'ingredients', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'preparation', 'make', 'a', 'choice', 'and', 'proceed', 'with', 'recipe', 'depending', 'on', 'size', 'of', 'squash', 'cut', 'into', 'half', 'or', 'fourths', 'remove', 'seeds', 'for', 'spicy', 'squash', 'drizzle', 'olive', 'oil', 'or', 'melted', 'butter', 'over', 'each', 'cut', 'squash', 'piece', 'season', 'with', 'mexican', 'seasoning', 'mix', 'ii', 'for', 'sweet', 'squash', 'drizzle', 'melted', 'honey', 'butter', 'grated', 'piloncillo', 'over', 'each', 'cut', 'squash', 'piece', 'season', 'with', 'sweet', 'mexican', 'spice', 'mix', 'bake', 'at', '350', 'degrees', 'again', 'depending', 'on', 'size', 'for', '40', 'minutes', 'up', 'to', 'an', 'hour', 'until', 'a', 'fork', 'can', 'easily', 'pierce', 'the', 'skin', 'be', 'careful', 'not', 'to', 

count    231637.000000
mean        120.858943
std          73.953188
min           8.000000
25%          72.000000
50%         105.000000
75%         150.000000
max        2195.000000
Name: token_count, dtype: float64

### 2.1.7 Word Frequency Analysis

We will analyze the most frequent tokens in the Food.com corpus to identify generic language, structural labels, and meaningful culinary vocabulary.

This analysis will guide the selection of stopwords while preserving terms that contribute to ingredient and recipe retrieval.

In [9]:
# --------------------------------------------------
# 2.1.7 Word Frequency Analysis
# --------------------------------------------------

from collections import Counter

# Count all tokens across the corpus
all_tokens = [
    token
    for recipe_tokens in recipes_df["tokens"]
    for token in recipe_tokens
]

word_frequencies = Counter(all_tokens)

print("Total tokens:", len(all_tokens))
print("Unique tokens:", len(word_frequencies))

# Most frequent 40 words
most_common_words = pd.DataFrame(
    word_frequencies.most_common(40),
    columns=["word", "frequency"]
)

display(most_common_words)

Total tokens: 27995403
Unique tokens: 59899


,word,frequency
0,and,1192411
1,the,1067499
2,in,606071
3,a,583362
4,to,576494
5,with,412389
6,until,345751
7,add,316044
8,ingredients,311159
9,of,308351


### 2.1.8 Stopword Analysis and Removal

Stopword removal can reduce uninformative language and improve the focus of text representations.

However, generic English stopword lists may remove terms that carry useful culinary meaning.

We will therefore compare the original tokens with a task-specific stopword-filtered version. Culinary actions, ingredient names, quantities, and preparation-related terms will be preserved.

In [10]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [11]:
# --------------------------------------------------
# 2.1.8 Stopword Removal
# --------------------------------------------------

from nltk.corpus import stopwords

# Standard English stopwords
english_stopwords = set(stopwords.words("english"))

# Structural labels introduced during text construction
structural_stopwords = {
    "recipe",
    "ingredients",
    "preparation"
}

# Task-specific stopword list
custom_stopwords = english_stopwords.union(structural_stopwords)

# Protect useful culinary terms if included in the generic list
protected_words = {
    "until",
    "over",
    "into",
    "after",
    "before",
    "up",
    "down",
    "out",
    "off",
    "under",
    "more",
    "less",
    "same",
    "each",
    "other"
}

custom_stopwords -= protected_words

# Remove selected stopwords
recipes_df["tokens_no_stopwords"] = recipes_df["tokens"].apply(
    lambda tokens: [
        token for token in tokens
        if token not in custom_stopwords
    ]
)

# Compare examples
for idx in [0, 1, 2]:

    print("=" * 80)
    print("Recipe:", recipes_df.loc[idx, "recipe_name"])

    print("\nOriginal tokens:")
    print(recipes_df.loc[idx, "tokens"][:50])

    print("\nFiltered tokens:")
    print(recipes_df.loc[idx, "tokens_no_stopwords"][:50])

    print("\nOriginal count:", len(recipes_df.loc[idx, "tokens"]))
    print(
        "Filtered count:",
        len(recipes_df.loc[idx, "tokens_no_stopwords"])
    )

print("\nNumber of stopwords:", len(custom_stopwords))

Recipe: arriba   baked winter squash mexican style

Original tokens:
['recipe', 'arriba', 'baked', 'winter', 'squash', 'mexican', 'style', 'ingredients', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'preparation', 'make', 'a', 'choice', 'and', 'proceed', 'with', 'recipe', 'depending', 'on', 'size', 'of', 'squash', 'cut', 'into', 'half', 'or', 'fourths', 'remove', 'seeds', 'for', 'spicy', 'squash', 'drizzle', 'olive', 'oil', 'or', 'melted', 'butter', 'over', 'each']

Filtered tokens:
['arriba', 'baked', 'winter', 'squash', 'mexican', 'style', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'make', 'choice', 'proceed', 'depending', 'size', 'squash', 'cut', 'into', 'half', 'fourths', 'remove', 'seeds', 'spicy', 'squash', 'drizzle', 'olive', 'oil', 'melted', 'butter', 'over', 'each', 'cut', 'squash', 'piece', 'season', 'mexican', 'seasoning', 'mix', 'ii', 'sweet', 'squash', 

In [12]:
# --------------------------------------------------
# 2.1.9 Refined Task-Specific Stopword List
# --------------------------------------------------

from nltk.corpus import stopwords

english_stopwords = set(stopwords.words("english"))

structural_stopwords = {
    "recipe",
    "ingredients",
    "preparation"
}

# Words whose meaning may matter in recipe instructions
protected_words = {
    "not",
    "no",
    "nor",
    "never",
    "without",
    "until",
    "over",
    "into",
    "after",
    "before",
    "up",
    "down",
    "out",
    "off",
    "under",
    "each",
    "more",
    "less"
}

custom_stopwords = (
    english_stopwords
    .union(structural_stopwords)
    .difference(protected_words)
)

recipes_df["tokens_no_stopwords"] = recipes_df["tokens"].apply(
    lambda tokens: [
        token for token in tokens
        if token not in custom_stopwords
    ]
)

print("Total stopwords:", len(custom_stopwords))
print("Protected words:", sorted(protected_words))

# Verify negation preservation
for phrase in ["do not bake", "without salt", "never add sugar"]:

    tokens = phrase.split()

    filtered = [
        token for token in tokens
        if token not in custom_stopwords
    ]

    print(f"\nOriginal: {phrase}")
    print(f"Filtered: {filtered}")

Total stopwords: 186
Protected words: ['after', 'before', 'down', 'each', 'into', 'less', 'more', 'never', 'no', 'nor', 'not', 'off', 'out', 'over', 'under', 'until', 'up', 'without']

Original: do not bake
Filtered: ['not', 'bake']

Original: without salt
Filtered: ['without', 'salt']

Original: never add sugar
Filtered: ['never', 'add', 'sugar']


In [13]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...


True

In [14]:
# --------------------------------------------------
# 2.1.10 Lemmatization Experiment
# --------------------------------------------------

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

test_words = [
    "tomatoes",
    "onions",
    "potatoes",
    "berries",
    "eggs",
    "chopped",
    "diced",
    "mixed",
    "baked",
    "cooked",
    "running",
    "leaves",
    "cheese",
    "ground"
]

for word in test_words:
    print(
        f"{word:12} -> "
        f"{lemmatizer.lemmatize(word):12} "
        f"(noun) | "
        f"{lemmatizer.lemmatize(word, pos='v'):12} "
        f"(verb)"
    )

tomatoes     -> tomato       (noun) | tomatoes     (verb)
onions       -> onion        (noun) | onions       (verb)
potatoes     -> potato       (noun) | potatoes     (verb)
berries      -> berry        (noun) | berry        (verb)
eggs         -> egg          (noun) | egg          (verb)
chopped      -> chopped      (noun) | chop         (verb)
diced        -> diced        (noun) | dice         (verb)
mixed        -> mixed        (noun) | mix          (verb)
baked        -> baked        (noun) | bake         (verb)
cooked       -> cooked       (noun) | cook         (verb)
running      -> running      (noun) | run          (verb)
leaves       -> leaf         (noun) | leave        (verb)
cheese       -> cheese       (noun) | cheese       (verb)
ground       -> ground       (noun) | grind        (verb)


In [15]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.


True

In [16]:
# --------------------------------------------------
# 2.1.11 POS-aware Lemmatization Experiment
# --------------------------------------------------

from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):

    if tag.startswith("J"):
        return "a"  # Adjective

    elif tag.startswith("V"):
        return "v"  # Verb

    elif tag.startswith("N"):
        return "n"  # Noun

    elif tag.startswith("R"):
        return "r"  # Adverb

    return "n"


test_sentence = [
    "ground", "beef", "with", "chopped", "onions",
    "baked", "potatoes", "and", "mixed", "vegetables"
]

tagged_words = pos_tag(test_sentence)

for word, tag in tagged_words:

    wordnet_pos = get_wordnet_pos(tag)

    lemma = lemmatizer.lemmatize(
        word,
        pos=wordnet_pos
    )

    print(f"{word:12} POS: {tag:6} Lemma: {lemma}")

ground       POS: NN     Lemma: ground
beef         POS: NN     Lemma: beef
with         POS: IN     Lemma: with
chopped      POS: JJ     Lemma: chopped
onions       POS: NNS    Lemma: onion
baked        POS: VBD    Lemma: bake
potatoes     POS: NNS    Lemma: potato
and          POS: CC     Lemma: and
mixed        POS: JJ     Lemma: mixed
vegetables   POS: NNS    Lemma: vegetable


In [17]:
# --------------------------------------------------
# 2.1.12 Conservative Lemmatization
# --------------------------------------------------

from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):

    tagged_tokens = pos_tag(tokens)

    lemmas = []

    for word, tag in tagged_tokens:

        if tag.startswith("V"):
            lemma = lemmatizer.lemmatize(word, pos="v")

        else:
            lemma = lemmatizer.lemmatize(word, pos="n")

        lemmas.append(lemma)

    return lemmas


# Apply to stopword-filtered tokens
recipes_df["tokens_lemmatized"] = recipes_df[
    "tokens_no_stopwords"
].apply(lemmatize_tokens)


# Compare representative recipes
for idx in [0, 1, 2]:

    print("=" * 80)
    print("Recipe:", recipes_df.loc[idx, "recipe_name"])

    print("\nBefore lemmatization:")
    print(recipes_df.loc[idx, "tokens_no_stopwords"][:50])

    print("\nAfter lemmatization:")
    print(recipes_df.loc[idx, "tokens_lemmatized"][:50])

Recipe: arriba   baked winter squash mexican style

Before lemmatization:
['arriba', 'baked', 'winter', 'squash', 'mexican', 'style', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'make', 'choice', 'proceed', 'depending', 'size', 'squash', 'cut', 'into', 'half', 'fourths', 'remove', 'seeds', 'spicy', 'squash', 'drizzle', 'olive', 'oil', 'melted', 'butter', 'over', 'each', 'cut', 'squash', 'piece', 'season', 'mexican', 'seasoning', 'mix', 'ii', 'sweet', 'squash', 'drizzle', 'melted']

After lemmatization:
['arriba', 'bake', 'winter', 'squash', 'mexican', 'style', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'make', 'choice', 'proceed', 'depend', 'size', 'squash', 'cut', 'into', 'half', 'fourth', 'remove', 'seed', 'spicy', 'squash', 'drizzle', 'olive', 'oil', 'melt', 'butter', 'over', 'each', 'cut', 'squash', 'piece', 'season', 'mexican', 'season', 'mix', 'ii', 'sweet', 

In [18]:
# --------------------------------------------------
# 2.1.13 Lemmatization Impact
# --------------------------------------------------

from collections import Counter

original_vocab = set(
    token
    for tokens in recipes_df["tokens_no_stopwords"]
    for token in tokens
)

lemmatized_vocab = set(
    token
    for tokens in recipes_df["tokens_lemmatized"]
    for token in tokens
)

print("Original vocabulary size:", len(original_vocab))
print("Lemmatized vocabulary size:", len(lemmatized_vocab))

print(
    "Vocabulary reduction:",
    round(
        (1 - len(lemmatized_vocab) / len(original_vocab)) * 100,
        2
    ),
    "%"
)

print("\nExamples of words no longer present:")

removed_words = sorted(original_vocab - lemmatized_vocab)

print(removed_words[:50])

Original vocabulary size: 59759
Lemmatized vocabulary size: 54975
Vocabulary reduction: 8.01 %

Examples of words no longer present:
['1000s', '100s', '2s', '30s', '3s', '4s', '50s', '5s', '60s', '6s', '70s', '7s', '8s', '90s', 'abought', 'abs', 'accelerated', 'accents', 'accepted', 'accepting', 'accessed', 'accessories', 'accidents', 'acclaimed', 'accolades', 'according', 'accordions', 'accounts', 'accouterments', 'accoutrements', 'accumulating', 'aces', 'aches', 'achieving', 'aching', 'acidifying', 'acknowledgments', 'acquainted', 'acress', 'acted', 'actions', 'activating', 'activities', 'acts', 'adams', 'adaptations', 'adapting', 'adas', 'adders', 'addicted']


In [19]:
# --------------------------------------------------
# 2.1.14 Inspect Common Culinary Spelling Variations
# --------------------------------------------------

spelling_variants = [
    "chili", "chilli",
    "cilantro", "coriander",
    "zucchini", "courgette",
    "eggplant", "aubergine",
    "yogurt", "yoghurt",
    "saute", "sautee",
    "barbecue", "barbeque",
    "ketchup", "catsup",
    "scallion", "shallot",
    "french", "francais"
]

for word in spelling_variants:
    print(f"{word:15} : {word_frequencies.get(word, 0):,}")

chili           : 26,390
chilli          : 1,821
cilantro        : 18,544
coriander       : 9,116
zucchini        : 13,010
courgette       : 156
eggplant        : 5,846
aubergine       : 562
yogurt          : 13,242
yoghurt         : 1,378
saute           : 24,386
sautee          : 950
barbecue        : 4,494
barbeque        : 322
ketchup         : 7,034
catsup          : 987
scallion        : 2,369
shallot         : 3,370
french          : 6,779
francais        : 5


In [20]:
# --------------------------------------------------
# 2.1.15 Controlled Spelling Normalization
# --------------------------------------------------

spelling_mapping = {
    "chilli": "chili",
    "yoghurt": "yogurt",
    "sautee": "saute",
    "barbeque": "barbecue",
    "catsup": "ketchup",
    "aubergine": "eggplant",
    "courgette": "zucchini"
}

# Apply normalization while preserving the original lemmas
recipes_df["tokens_normalized"] = recipes_df[
    "tokens_lemmatized"
].apply(
    lambda tokens: [
        spelling_mapping.get(token, token)
        for token in tokens
    ]
)

# Verify selected transformations
for original, normalized in spelling_mapping.items():

    print(
        f"{original:12} -> {normalized:12}"
    )

# Check examples in the resulting dataset
for idx in [0, 1, 2]:

    print("\nRecipe:", recipes_df.loc[idx, "recipe_name"])
    print(recipes_df.loc[idx, "tokens_normalized"][:30])

chilli       -> chili       
yoghurt      -> yogurt      
sautee       -> saute       
barbeque     -> barbecue    
catsup       -> ketchup     
aubergine    -> eggplant    
courgette    -> zucchini    

Recipe: arriba   baked winter squash mexican style
['arriba', 'bake', 'winter', 'squash', 'mexican', 'style', 'winter', 'squash', 'mexican', 'seasoning', 'mixed', 'spice', 'honey', 'butter', 'olive', 'oil', 'salt', 'make', 'choice', 'proceed', 'depend', 'size', 'squash', 'cut', 'into', 'half', 'fourth', 'remove', 'seed', 'spicy']

Recipe: a bit different  breakfast pizza
['bit', 'different', 'breakfast', 'pizza', 'prepare', 'pizza', 'crust', 'sausage', 'patty', 'egg', 'milk', 'salt', 'pepper', 'cheese', 'preheat', 'oven', '425', 'degree', 'f', 'press', 'dough', 'into', 'bottom', 'side', '12', 'inch', 'pizza', 'pan', 'bake', '5']

Recipe: all in the kitchen  chili
['kitchen', 'chili', 'ground', 'beef', 'yellow', 'onion', 'dice', 'tomato', 'tomato', 'paste', 'tomato', 'soup', 'rotel', 't

In [21]:
# --------------------------------------------------
# 2.1.16 Verify Spelling Normalization
# --------------------------------------------------

from collections import Counter

normalized_frequencies = Counter(
    token
    for tokens in recipes_df["tokens_normalized"]
    for token in tokens
)

print("Vocabulary before spelling normalization:",
      len(lemmatized_vocab))

print("Vocabulary after spelling normalization:",
      len(set(normalized_frequencies.keys())))

print("\nVerification of normalized terms:")

for original, canonical in spelling_mapping.items():

    print(
        f"{original:12} : {normalized_frequencies.get(original, 0):,}"
    )

    print(
        f"{canonical:12} : {normalized_frequencies.get(canonical, 0):,}"
    )

    print("-" * 35)

Vocabulary before spelling normalization: 54975
Vocabulary after spelling normalization: 54968

Verification of normalized terms:
chilli       : 0
chili        : 29,353
-----------------------------------
yoghurt      : 0
yogurt       : 14,629
-----------------------------------
sautee       : 0
saute        : 26,640
-----------------------------------
barbeque     : 0
barbecue     : 5,416
-----------------------------------
catsup       : 0
ketchup      : 8,022
-----------------------------------
aubergine    : 0
eggplant     : 7,627
-----------------------------------
courgette    : 0
zucchini     : 13,338
-----------------------------------


In [22]:
# --------------------------------------------------
# 2.1.17 Final Processed Text Representation
# --------------------------------------------------

recipes_df["processed_text"] = recipes_df[
    "tokens_normalized"
].apply(lambda tokens: " ".join(tokens))

# Inspect representative examples
for idx in [0, 1, 2]:

    print("=" * 80)
    print("Recipe:", recipes_df.loc[idx, "recipe_name"])

    print("\nOriginal text:")
    print(recipes_df.loc[idx, "recipe_text_core"][:400])

    print("\nProcessed text:")
    print(recipes_df.loc[idx, "processed_text"][:400])

    print(
        "\nProcessed token count:",
        len(recipes_df.loc[idx, "tokens_normalized"])
    )

print("\nMissing processed texts:",
      recipes_df["processed_text"].isna().sum())

print("Empty processed texts:",
      recipes_df["processed_text"].str.strip().eq("").sum())

Recipe: arriba   baked winter squash mexican style

Original text:
Recipe: arriba   baked winter squash mexican style. Ingredients: winter squash mexican seasoning mixed spice honey butter olive oil salt. Preparation: make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzl

Processed text:
arriba bake winter squash mexican style winter squash mexican seasoning mixed spice honey butter olive oil salt make choice proceed depend size squash cut into half fourth remove seed spicy squash drizzle olive oil melt butter over each cut squash piece season mexican season mix ii sweet squash drizzle melt honey butter grate piloncillo over each cut squash piece season sweet mexican spice mix bak

Processed token count: 103
Recipe: a bit different  breakfast pizza

Original text:
Recipe: a bit different  brea

In [23]:
# --------------------------------------------------
# 2.1.18 Save NLP Preprocessing Checkpoint
# --------------------------------------------------

# Stopword-only representation
recipes_df["text_no_stopwords"] = recipes_df[
    "tokens_no_stopwords"
].apply(lambda tokens: " ".join(tokens))

# Keep only the columns needed for future NLP experiments
nlp_checkpoint = recipes_df[
    [
        "recipe_id",
        "recipe_name",
        "normalized_text",
        "text_no_stopwords",
        "processed_text"
    ]
].copy()

# Output path
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "recipes"
    / "foodcom_nlp_preprocessed.csv.gz"
)

# Save compressed CSV
nlp_checkpoint.to_csv(
    OUTPUT_PATH,
    index=False,
    compression="gzip"
)

print("Checkpoint saved successfully!")
print("Shape:", nlp_checkpoint.shape)
print("Path:", OUTPUT_PATH)
print("File size (MB):", round(OUTPUT_PATH.stat().st_size / (1024**2), 2))

Checkpoint saved successfully!
Shape: (231637, 5)
Path: C:\Users\dell\NutriVision\data\processed\recipes\foodcom_nlp_preprocessed.csv.gz
File size (MB): 71.09


In [24]:
# --------------------------------------------------
# 2.2.1 Load Preprocessed Dataset
# --------------------------------------------------

nlp_checkpoint = pd.read_csv(
    OUTPUT_PATH,
    compression="gzip"
)

print("Dataset shape:", nlp_checkpoint.shape)
print("\nColumns:")
print(nlp_checkpoint.columns.tolist())

print("\nFirst three recipes:")
display(
    nlp_checkpoint[
        ["recipe_id", "recipe_name", "processed_text"]
    ].head(3)
)

Dataset shape: (231637, 5)

Columns:
['recipe_id', 'recipe_name', 'normalized_text', 'text_no_stopwords', 'processed_text']

First three recipes:


,recipe_id,recipe_name,processed_text
0,137739,arriba baked winter squash mexican style,arriba bake winter squash mexican style winter...
1,31490,a bit different breakfast pizza,bit different breakfast pizza prepare pizza cr...
2,112140,all in the kitchen chili,kitchen chili ground beef yellow onion dice to...


In [25]:
# --------------------------------------------------
# 2.2.2.1 Bag of Words - Small Example
# --------------------------------------------------

from sklearn.feature_extraction.text import CountVectorizer

sample_recipes = [
    "chicken rice chicken tomato",
    "rice beef tomato",
    "chicken beef rice"
]

bow_demo = CountVectorizer()

bow_matrix = bow_demo.fit_transform(sample_recipes)

bow_demo_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_demo.get_feature_names_out(),
    index=["Recipe A", "Recipe B", "Recipe C"]
)

display(bow_demo_df)

print("Matrix shape:", bow_matrix.shape)

,beef,chicken,rice,tomato
Recipe A,0,2,1,1
Recipe B,1,0,1,1
Recipe C,1,1,1,0


Matrix shape: (3, 4)


In [26]:
# --------------------------------------------------
# 2.2.2.2 Full Food.com Bag of Words
# --------------------------------------------------

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

bow_vectorizer = CountVectorizer(
    dtype=np.int32,
    min_df=1
)

bow_matrix = bow_vectorizer.fit_transform(
    nlp_checkpoint["processed_text"]
)

# Basic information
print("BoW matrix shape:", bow_matrix.shape)

print(
    "Vocabulary size:",
    len(bow_vectorizer.get_feature_names_out())
)

print("Non-zero entries:", f"{bow_matrix.nnz:,}")

print(
    "Matrix density:",
    f"{(bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100:.5f}%"
)

# Actual sparse matrix memory usage
matrix_memory = (
    bow_matrix.data.nbytes
    + bow_matrix.indices.nbytes
    + bow_matrix.indptr.nbytes
)

print(
    "Sparse matrix memory:",
    round(matrix_memory / (1024 ** 2), 2),
    "MB"
)

BoW matrix shape: (231637, 54934)
Vocabulary size: 54934
Non-zero entries: 12,784,442
Matrix density: 0.10047%
Sparse matrix memory: 98.42 MB


In [27]:
# --------------------------------------------------
# 2.2.3 Inspect BoW Recipe Vectors
# --------------------------------------------------

feature_names = bow_vectorizer.get_feature_names_out()

for idx in [0, 1, 2]:

    recipe_vector = bow_matrix[idx]

    nonzero_indices = recipe_vector.indices
    word_counts = recipe_vector.data

    recipe_features = pd.DataFrame({
        "word": feature_names[nonzero_indices],
        "frequency": word_counts
    }).sort_values(
        by="frequency",
        ascending=False
    )

    print("=" * 70)
    print("Recipe:", nlp_checkpoint.loc[idx, "recipe_name"])
    print("Number of unique words:", len(nonzero_indices))

    display(recipe_features.head(20))

Recipe: arriba   baked winter squash mexican style
Number of unique words: 66


,word,frequency
3,squash,9
4,mexican,4
10,butter,4
31,season,4
19,cut,3
1,bake,2
32,mix,2
12,oil,2
8,spice,2
2,winter,2


Recipe: a bit different  breakfast pizza
Number of unique words: 47


,word,frequency
16,sausage,4
18,egg,4
8,until,3
21,cheese,3
15,crust,3
13,pizza,3
0,bake,2
4,over,2
33,brown,2
32,set,2


Recipe: all in the kitchen  chili
Number of unique words: 40


,word,frequency
7,ground,4
12,tomato,4
8,beef,3
6,chili,3
25,add,3
17,bean,3
1,hour,2
3,cheese,2
16,kidney,2
21,lettuce,2


In [28]:
# --------------------------------------------------
# 2.2.4 Full Food.com TF-IDF Representation
# --------------------------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

tfidf_vectorizer = TfidfVectorizer(
    dtype=np.float32,
    min_df=1,
    norm="l2"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    nlp_checkpoint["processed_text"]
)

# Basic information
print("TF-IDF matrix shape:", tfidf_matrix.shape)

print(
    "Vocabulary size:",
    len(tfidf_vectorizer.get_feature_names_out())
)

print("Non-zero entries:", f"{tfidf_matrix.nnz:,}")

print(
    "Matrix density:",
    f"{(tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.5f}%"
)

# Sparse matrix memory
tfidf_memory = (
    tfidf_matrix.data.nbytes
    + tfidf_matrix.indices.nbytes
    + tfidf_matrix.indptr.nbytes
)

print(
    "Sparse matrix memory:",
    round(tfidf_memory / (1024 ** 2), 2),
    "MB"
)

TF-IDF matrix shape: (231637, 54934)
Vocabulary size: 54934
Non-zero entries: 12,784,442
Matrix density: 0.10047%
Sparse matrix memory: 98.42 MB


In [29]:
# --------------------------------------------------
# 2.2.5 Compare BoW and TF-IDF
# --------------------------------------------------

bow_features = bow_vectorizer.get_feature_names_out()
tfidf_features = tfidf_vectorizer.get_feature_names_out()

for idx in [0, 1, 2]:

    bow_row = bow_matrix.getrow(idx)
    tfidf_row = tfidf_matrix.getrow(idx)

    comparison = pd.DataFrame({
        "word": bow_features[bow_row.indices],
        "BoW_count": bow_row.data,
        "TF-IDF_weight": tfidf_row.data
    })

    comparison = comparison.sort_values(
        by="TF-IDF_weight",
        ascending=False
    )

    print("=" * 70)
    print("Recipe:", nlp_checkpoint.loc[idx, "recipe_name"])

    display(comparison.head(20))

Recipe: arriba   baked winter squash mexican style


,word,BoW_count,TF-IDF_weight
3,squash,9,0.677517
4,mexican,4,0.297188
2,winter,2,0.203007
0,arriba,1,0.165391
31,season,4,0.155389
36,piloncillo,1,0.151410
17,depend,2,0.141468
57,comfortable,1,0.130209
52,opt,1,0.129419
10,butter,4,0.118222


Recipe: a bit different  breakfast pizza


,word,BoW_count,TF-IDF_weight
16,sausage,4,0.484680
13,pizza,3,0.411035
15,crust,3,0.320568
18,egg,4,0.242330
21,cheese,3,0.179411
11,different,1,0.169787
37,frothy,1,0.164961
39,baked,1,0.153826
12,breakfast,1,0.151230
19,milk,2,0.139565


Recipe: all in the kitchen  chili


,word,BoW_count,TF-IDF_weight
16,kidney,2,0.309477
12,tomato,4,0.306774
17,bean,3,0.304482
6,chili,3,0.299850
8,beef,3,0.286242
7,ground,4,0.274508
21,lettuce,2,0.254482
29,wilted,1,0.212327
15,rotel,1,0.186890
5,kitchen,1,0.156707


In [30]:
# --------------------------------------------------
# 2.2.6 Word Unigrams + Bigrams TF-IDF
# --------------------------------------------------

word_ngram_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    dtype=np.float32,
    min_df=2,
    norm="l2"
)

word_ngram_matrix = word_ngram_vectorizer.fit_transform(
    nlp_checkpoint["processed_text"]
)

# Basic information
print("Word n-gram matrix shape:", word_ngram_matrix.shape)

print(
    "Vocabulary size:",
    len(word_ngram_vectorizer.get_feature_names_out())
)

print("Non-zero entries:", f"{word_ngram_matrix.nnz:,}")

print(
    "Matrix density:",
    f"{(word_ngram_matrix.nnz / (word_ngram_matrix.shape[0] * word_ngram_matrix.shape[1])) * 100:.5f}%"
)

ngram_memory = (
    word_ngram_matrix.data.nbytes
    + word_ngram_matrix.indices.nbytes
    + word_ngram_matrix.indptr.nbytes
)

print(
    "Sparse matrix memory:",
    round(ngram_memory / (1024 ** 2), 2),
    "MB"
)

Word n-gram matrix shape: (231637, 593733)
Vocabulary size: 593733
Non-zero entries: 29,772,890
Matrix density: 0.02165%
Sparse matrix memory: 228.03 MB


In [31]:
# --------------------------------------------------
# 2.2.7 Inspect Word N-grams
# --------------------------------------------------

ngram_features = word_ngram_vectorizer.get_feature_names_out()

for idx in [0, 1, 2]:

    row = word_ngram_matrix.getrow(idx)

    feature_df = pd.DataFrame({
        "feature": ngram_features[row.indices],
        "tfidf_weight": row.data
    }).sort_values(
        by="tfidf_weight",
        ascending=False
    )

    bigrams = feature_df[
        feature_df["feature"].str.contains(" ")
    ]

    print("=" * 70)
    print("Recipe:", nlp_checkpoint.loc[idx, "recipe_name"])

    print("\nTop 15 bigrams:")
    display(bigrams.head(15))

    print(
        "Total non-zero bigrams:",
        len(bigrams)
    )

Recipe: arriba   baked winter squash mexican style

Top 15 bigrams:


,feature,tfidf_weight
68,squash mexican,0.204126
90,squash drizzle,0.176242
98,squash piece,0.165771
67,winter squash,0.144589
96,each cut,0.142267
99,piece season,0.138523
97,cut squash,0.129993
74,honey butter,0.121692
80,depend size,0.110146
130,squash especially,0.102063


Total non-zero bigrams: 83
Recipe: a bit different  breakfast pizza

Top 15 bigrams:


,feature,tfidf_weight
48,different breakfast,0.172233
89,frothy spoon,0.165054
53,crust sausage,0.155311
47,bit different,0.155311
55,patty egg,0.141882
64,degree press,0.138111
49,breakfast pizza,0.137704
84,piece whisk,0.137704
56,egg milk,0.133633
90,spoon sausage,0.132413


Total non-zero bigrams: 65
Recipe: all in the kitchen  chili

Top 15 bigrams:


,feature,tfidf_weight
41,ground beef,0.184993
91,cold clean,0.174182
92,clean lettuce,0.166036
53,kidney bean,0.165425
71,beef almost,0.147507
79,bean like,0.146387
80,like bean,0.146387
73,brown saute,0.145858
93,lettuce shred,0.143914
60,salt lettuce,0.143466


Total non-zero bigrams: 55


In [32]:
# --------------------------------------------------
# 2.2.8.1 Character N-gram TF-IDF
# --------------------------------------------------

char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=5,
    max_features=150000,
    dtype=np.float32,
    norm="l2"
)

char_matrix = char_vectorizer.fit_transform(
    nlp_checkpoint["processed_text"]
)

# Basic information
print("Character n-gram matrix shape:", char_matrix.shape)

print(
    "Vocabulary size:",
    len(char_vectorizer.get_feature_names_out())
)

print("Non-zero entries:", f"{char_matrix.nnz:,}")

print(
    "Matrix density:",
    f"{(char_matrix.nnz / (char_matrix.shape[0] * char_matrix.shape[1])) * 100:.5f}%"
)

char_memory = (
    char_matrix.data.nbytes
    + char_matrix.indices.nbytes
    + char_matrix.indptr.nbytes
)

print(
    "Sparse matrix memory:",
    round(char_memory / (1024 ** 2), 2),
    "MB"
)

Character n-gram matrix shape: (231637, 70079)
Vocabulary size: 70079
Non-zero entries: 145,079,472
Matrix density: 0.89374%
Sparse matrix memory: 1107.75 MB


In [33]:
# --------------------------------------------------
# 2.2.9 Inspect Character N-grams
# --------------------------------------------------

char_features = char_vectorizer.get_feature_names_out()

for idx in [0, 1, 2]:

    row = char_matrix.getrow(idx)

    feature_df = pd.DataFrame({
        "character_ngram": char_features[row.indices],
        "tfidf_weight": row.data
    }).sort_values(
        by="tfidf_weight",
        ascending=False
    )

    print("=" * 70)
    print("Recipe:", nlp_checkpoint.loc[idx, "recipe_name"])

    display(feature_df.head(20))

Recipe: arriba   baked winter squash mexican style


,character_ngram,tfidf_weight
52,uash,0.216439
50,squas,0.216352
51,quash,0.216352
47,uash,0.216352
46,quas,0.216308
41,uas,0.215959
49,squa,0.160215
45,squa,0.160180
44,squ,0.145568
38,sq,0.145482


Recipe: a bit different  breakfast pizza


,character_ngram,tfidf_weight
229,ausag,0.153081
228,sausa,0.153070
223,ausa,0.153065
224,usag,0.153043
230,usage,0.153043
227,saus,0.152915
222,saus,0.152883
217,usa,0.150776
216,aus,0.149499
231,sage,0.141313


Recipe: all in the kitchen  chili


,character_ngram,tfidf_weight
117,ki,0.113414
245,ean,0.103308
234,kidn,0.099957
239,idney,0.099957
237,kidn,0.099957
238,kidne,0.099957
235,idne,0.099949
240,dney,0.099908
236,dney,0.099900
231,idn,0.099468


In [35]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Python executable: C:\Users\dell\NutriVision\.venv\Scripts\python.exe
